# Session 3: building arithmetic out of gates
**A. Real-world problem.** Inside a processor there is no "plus" button, only gates. Yet it adds
very many numbers every second, subtracts, and knows when an answer does not fit. We build that
from scratch: a tiny simulator where a circuit is a list of gates, then a half adder, a full
adder, a 4-bit ripple-carry adder, subtraction by two's complement, and a 4-bit ALU slice. Every
circuit is checked against Python's own integer arithmetic for every possible input.

**Plan.** (1) the simulator, (2) adders, (3) negative numbers and overflow, (4) how long the
carry takes to ripple (gate delays), (5) the ALU, (6) a seven-segment decoder as a stretch.
This is a simplified educational model: ideal gates, each with the same delay of 1 unit, no
wires, no clock and no real chip timing.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: a gate-level simulator
A *wire* is a name. A *gate* reads one or two wires and creates a new wire. Values are numpy
arrays of 0/1, so one `run` tests thousands of input combinations at once. Each wire also has a
*depth*: 0 for an input, and 1 + the largest depth of its inputs for a gate output. Depth counts
gate delays on the longest path to that wire.

In [ ]:
OPS = {
    "AND": lambda a, b: a & b,
    "OR": lambda a, b: a | b,
    "XOR": lambda a, b: a ^ b,
    "NAND": lambda a, b: 1 - (a & b),
    "NOT": lambda a: 1 - a,
}


class Circuit:
    def __init__(self):
        self.gates = []          # (output wire, op, input wires) in the order they were added
        self.depth = {}          # wire name -> gate delays from the inputs

    def input(self, name):
        self.depth[name] = 0
        return name

    def gate(self, op, *ins):
        out = f"w{len(self.gates)}"
        self.gates.append((out, op, ins))
        self.depth[out] = 1 + max(self.depth[i] for i in ins)
        return out

    def run(self, values):
        """values: dict input name -> array of 0/1. Returns every wire's array."""
        wires = {k: np.asarray(v, dtype=np.int64) for k, v in values.items()}
        for out, op, ins in self.gates:
            wires[out] = OPS[op](*[wires[i] for i in ins])
        return wires

    def size(self):
        return len(self.gates)

## Step 2: half adder, full adder, ripple-carry adder
Adding two bits: the *sum* bit is 1 when the inputs differ (XOR) and the *carry* is 1 when both
are 1 (AND). That is a **half adder**. A **full adder** also takes a carry in: two half adders and
an OR. Chain n full adders, carry out of one into the next, and you have an n-bit
**ripple-carry adder**.

In [ ]:
def half_adder(c, a, b):
    return c.gate("XOR", a, b), c.gate("AND", a, b)          # sum, carry


def full_adder(c, a, b, cin):
    s1, c1 = half_adder(c, a, b)
    s, c2 = half_adder(c, s1, cin)
    return s, c.gate("OR", c1, c2)                           # sum, carry out


def ripple_adder(c, a_bits, b_bits, cin):
    """a_bits, b_bits: wire names, least significant bit first. Returns (sum wires, carry-out wires)."""
    sums, carries, carry = [], [], cin
    for a, b in zip(a_bits, b_bits):
        s, carry = full_adder(c, a, b, carry)
        sums.append(s)
        carries.append(carry)
    return sums, carries


def build_adder(n):
    """An n-bit adder with inputs a0..a(n-1), b0..b(n-1), cin. Returns a dict describing it."""
    c = Circuit()
    a = [c.input(f"a{i}") for i in range(n)]
    b = [c.input(f"b{i}") for i in range(n)]
    cin = c.input("cin")
    sums, carries = ripple_adder(c, a, b, cin)
    return {"c": c, "n": n, "a": a, "b": b, "cin": cin, "sum": sums, "carry": carries}


def bits_of(values, n):
    """Array of unsigned numbers -> list of n arrays (bit 0 first)."""
    values = np.asarray(values, dtype=np.int64)
    return [(values >> i) & 1 for i in range(n)]


def number_of(bit_arrays):
    """List of bit arrays (bit 0 first) -> array of unsigned numbers."""
    return sum(b.astype(np.int64) << i for i, b in enumerate(bit_arrays))


def add_bits(adder, a, b, cin=0):
    """Run a built adder on arrays of unsigned numbers a, b; returns (sum numbers, carry-out bits)."""
    n = adder["n"]
    a, b = np.broadcast_arrays(np.asarray(a), np.asarray(b))
    vals = {adder["cin"]: np.full(a.shape, cin)}
    for i, (x, y) in enumerate(zip(bits_of(a, n), bits_of(b, n))):
        vals[adder["a"][i]], vals[adder["b"][i]] = x, y
    w = adder["c"].run(vals)
    return number_of([w[s] for s in adder["sum"]]), w[adder["carry"][-1]]

## Step 3: negative numbers, subtraction and overflow
With n bits, *two's complement* reads the pattern u as u if u < 2^(n-1), else u - 2^n. So 4 bits
cover -8 to 7, and 1111 means -1. To negate: flip every bit and add 1. Then a - b is just
a + (NOT b) + 1, so one adder does both jobs: XOR every b bit with a `sub` control and feed `sub`
in as the carry in. *Overflow* (the true answer does not fit) happens exactly when the carry into the
top bit differs from the carry out of it.

In [ ]:
def to_twos(v, n):
    """Signed integer -> n-bit pattern (an unsigned number)."""
    return v % (2 ** n)


def from_twos(u, n):
    """n-bit pattern -> signed integer."""
    u = np.asarray(u)
    return np.where(u >= 2 ** (n - 1), u - 2 ** n, u)


def build_add_sub(n):
    """n-bit adder-subtractor: sub = 0 gives a + b, sub = 1 gives a - b. Has an overflow flag (n >= 2)."""
    c = Circuit()
    a = [c.input(f"a{i}") for i in range(n)]
    b = [c.input(f"b{i}") for i in range(n)]
    sub = c.input("sub")
    bx = [c.gate("XOR", bi, sub) for bi in b]
    sums, carries = ripple_adder(c, a, bx, sub)
    overflow = c.gate("XOR", carries[n - 2], carries[n - 1])
    return {"c": c, "n": n, "a": a, "b": b, "sub": sub, "sum": sums, "carry": carries, "overflow": overflow}


def add_sub(circ, a, b, sub):
    """a, b: arrays of signed integers; sub: 0 or 1. Returns (signed result, overflow flag)."""
    n = circ["n"]
    ua, ub = to_twos(np.asarray(a), n), to_twos(np.asarray(b), n)
    ua, ub = np.broadcast_arrays(ua, ub)
    vals = {circ["sub"]: np.full(ua.shape, sub)}
    for i, (x, y) in enumerate(zip(bits_of(ua, n), bits_of(ub, n))):
        vals[circ["a"][i]], vals[circ["b"][i]] = x, y
    w = circ["c"].run(vals)
    return from_twos(number_of([w[s] for s in circ["sum"]]), n), w[circ["overflow"]]


def python_reference(a, b, sub, n):
    """What Python says: (result wrapped into n bits as a signed number, true result did not fit)."""
    true = a - b if sub else a + b
    wrapped = int(from_twos(to_twos(true, n), n))
    return wrapped, int(true != wrapped)

## Step 4: how long does the carry take?
Each gate takes one unit of delay. In a ripple adder the carry out of stage i is needed by stage
i+1, so the delay of the last carry grows with the number of bits. We read it off the
simulator's `depth`. This is why real processors use faster carry circuits (carry-lookahead,
a stretch topic) instead of rippling.

In [ ]:
def carry_delay(adder):
    """Gate delays until the last carry out is valid (longest path)."""
    return adder["c"].depth[adder["carry"][-1]]


def sum_delay(adder):
    return max(adder["c"].depth[s] for s in adder["sum"])


def longest_carry_run(a, b, n):
    """Longest run of stages in a row that pass a carry on (a data-dependent chain), for unsigned a + b."""
    carry, run, best = 0, 0, 0
    for i in range(n):
        x, y = (a >> i) & 1, (b >> i) & 1
        carry = (x & y) | ((x ^ y) & carry)
        run = run + 1 if carry else 0
        best = max(best, run)
    return best

## Step 5: a 4-bit ALU slice
An ALU (arithmetic logic unit) does several operations on the same inputs and a 2-bit control
code picks which one comes out: 00 add, 01 subtract, 10 AND, 11 OR. We select with AND gates
and an OR, all inside the simulator.

In [ ]:
OP_ADD, OP_SUB, OP_AND, OP_OR = 0, 1, 2, 3


def build_alu(n=4):
    c = Circuit()
    a = [c.input(f"a{i}") for i in range(n)]
    b = [c.input(f"b{i}") for i in range(n)]
    op1, op0 = c.input("op1"), c.input("op0")
    not1, not0 = c.gate("NOT", op1), c.gate("NOT", op0)
    sub = c.gate("AND", not1, op0)                  # 01
    sel_and = c.gate("AND", op1, not0)              # 10
    sel_or = c.gate("AND", op1, op0)                # 11
    bx = [c.gate("XOR", bi, sub) for bi in b]
    sums, carries = ripple_adder(c, a, bx, sub)
    ovf = c.gate("AND", c.gate("XOR", carries[n - 2], carries[n - 1]), not1)
    out = []
    for i in range(n):
        arith = c.gate("AND", not1, sums[i])
        land = c.gate("AND", sel_and, c.gate("AND", a[i], b[i]))
        lor = c.gate("AND", sel_or, c.gate("OR", a[i], b[i]))
        out.append(c.gate("OR", c.gate("OR", arith, land), lor))
    return {"c": c, "n": n, "a": a, "b": b, "op1": op1, "op0": op0, "out": out, "overflow": ovf}


def run_alu(alu, a, b, op):
    """a, b: unsigned patterns (0 to 2^n - 1), op: 0..3. Returns (result pattern, overflow flag)."""
    n = alu["n"]
    a, b = np.broadcast_arrays(np.asarray(a), np.asarray(b))
    vals = {alu["op1"]: np.full(a.shape, op >> 1), alu["op0"]: np.full(a.shape, op & 1)}
    for i, (x, y) in enumerate(zip(bits_of(a, n), bits_of(b, n))):
        vals[alu["a"][i]], vals[alu["b"][i]] = x, y
    w = alu["c"].run(vals)
    return number_of([w[o] for o in alu["out"]]), w[alu["overflow"]]


def alu_reference(a, b, op, n):
    """Python's answer for unsigned patterns a, b: (result pattern, signed overflow flag)."""
    mask = 2 ** n - 1
    if op in (OP_ADD, OP_SUB):
        sa, sb = int(from_twos(a, n)), int(from_twos(b, n))
        true = sa + sb if op == OP_ADD else sa - sb
        return (a + b) & mask if op == OP_ADD else (a - b) & mask, int(not -(2 ** (n - 1)) <= true < 2 ** (n - 1))
    return (a & b if op == OP_AND else a | b), 0

## Stretch: a seven-segment decoder
A digit display lights segments a to g. The table below is a *truth table with 4 inputs* (the bits
of the digit) and 7 outputs; each output could be built with the sum-of-products method.

In [ ]:
SEGMENTS = {
    0: "abcdef", 1: "bc", 2: "abdeg", 3: "abcdg", 4: "bcfg",
    5: "acdfg", 6: "acdefg", 7: "abc", 8: "abcdefg", 9: "abcdfg",
}


def seven_segment(d):
    """Seven 0/1 outputs (a, b, c, d, e, f, g) for digit 0 to 9."""
    return [1 if s in SEGMENTS[d] else 0 for s in "abcdefg"]

## Step 6: see it
Left: the carry delay against the number of bits, read from the simulator. Right: the failure
case, 7 + 1 in 4 bits, with the overflow flag.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    widths = np.arange(2, 17)
    delays = [carry_delay(build_adder(int(n))) for n in widths]
    for n in (4, 8, 16):
        ad = build_adder(n)
        print(f"{n:2d}-bit adder: {ad['c'].size()} gates, carry-out depth {carry_delay(ad)}, last sum depth {sum_delay(ad)}")
    fig, ax = plt.subplots(figsize=(6, 3.8))
    ax.plot(widths, delays, "o-", color="#c0561b")
    ax.set_xlabel("adder width (bits)")
    ax.set_ylabel("carry-out delay (gate delays)")
    ax.set_title("Ripple-carry delay grows in step with the width")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "carry_delay.png", dpi=120)

    circ = build_add_sub(4)
    A, B = np.meshgrid(np.arange(-8, 8), np.arange(-8, 8))
    res, ovf = add_sub(circ, A.ravel(), B.ravel(), 0)
    fig2, ax2 = plt.subplots(figsize=(5.2, 4.4))
    ax2.imshow(ovf.reshape(16, 16), cmap="YlOrBr", origin="lower", extent=(-8.5, 7.5, -8.5, 7.5), vmin=0, vmax=1.4)
    ax2.set_xlabel("a (signed, 4 bits)")
    ax2.set_ylabel("b (signed, 4 bits)")
    ax2.set_title("Overflow flag for a + b (dark = overflow)")
    fig2.tight_layout()
    fig2.savefig(OUT / "overflow_map.png", dpi=120)
    r, o = add_sub(circ, 7, 1, 0)
    print(f"failure case: 7 + 1 in 4 bits gives {int(r)} with overflow flag {int(o)}")
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Print the gate count and the carry delay of `build_adder(n)` for n = 1, 2, 3, 4. Predict each from 5n and 2n + 1 first.
2. **Stretch.** Use `add_sub` to find all pairs of 4-bit signed numbers (a, b) with a - b overflowing, and count them. Plot them like the overflow map.
3. **Explain.** Stretch: build a seven-segment decoder segment 'a' with sum of products from `SEGMENTS`: list which digits light segment a, make the minterms of the 4-bit digit code, and check your circuit against `seven_segment` for the digits 0 to 9.

## Assessment
- **A. Conceptual.** Why does 7 + 1 give -8 on four bits, and how does the circuit know that the answer is wrong?
- **B. Numerical.** Compute 5 - 7 on four bits: show the three numbers you add, the four-bit result, and whether overflow occurs. Sanity check with ordinary arithmetic.
- **C. Graph and simulation interpretation.** From the delay graph, how many gate delays does the carry take for 8 bits, and for 12 bits? What does the slope of the line tell you?
- **D. Design and optimisation challenge.** Design: each gate takes 0.5 ns and the carry out of a ripple adder must be ready within 20 ns. What is the widest adder allowed? What would you change to go wider?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []

    # half adder and full adder, every row
    c = L.Circuit()
    a, b, ci = c.input("a"), c.input("b"), c.input("ci")
    hs, hc = L.half_adder(c, a, b)
    fs, fc = L.full_adder(c, a, b, ci)
    rows = np.array([[x, y, z] for x in (0, 1) for y in (0, 1) for z in (0, 1)])
    w = c.run({"a": rows[:, 0], "b": rows[:, 1], "ci": rows[:, 2]})
    checks.append(("half adder: sum = XOR, carry = AND on all 4 input pairs",
                   np.array_equal(2 * w[hc] + w[hs], rows[:, 0] + rows[:, 1])))
    checks.append(("full adder: 2*carry + sum = a + b + cin on all 8 rows",
                   np.array_equal(2 * w[fc] + w[fs], rows.sum(axis=1))))

    # 4-bit ripple adder, every pair and both carry-ins
    ad = L.build_adder(4)
    A, B = np.meshgrid(np.arange(16), np.arange(16))
    A, B = A.ravel(), B.ravel()
    good = True
    for cin in (0, 1):
        s, co = L.add_bits(ad, A, B, cin)
        total = A + B + cin
        good &= np.array_equal(s, total % 16) and np.array_equal(co, total // 16)
    checks.append(("4-bit ripple adder matches Python for all 256 pairs, carry in 0 and 1", bool(good)))
    checks.append(("a 4-bit adder uses 20 gates (5 per full adder), an 8-bit adder 40",
                   ad["c"].size() == 20 and L.build_adder(8)["c"].size() == 40))

    # two's complement
    checks.append(("two's complement: 4-bit patterns 8 -> -8, 15 -> -1, 7 -> 7, and -3 is 1101",
                   int(L.from_twos(8, 4)) == -8 and int(L.from_twos(15, 4)) == -1 and int(L.from_twos(7, 4)) == 7
                   and L.to_twos(-3, 4) == 13 and format(L.to_twos(-3, 4), "04b") == "1101"))
    flip_plus_one = [(~u + 1) & 15 for u in range(16)]
    checks.append(("negating = flip every bit and add 1, for all 16 patterns (except -8 which maps to itself)",
                   all(int(L.from_twos(flip_plus_one[u], 4)) == -int(L.from_twos(u, 4)) for u in range(16) if u != 8)
                   and flip_plus_one[8] == 8))
    # worked example 6 - 3
    circ = L.build_add_sub(4)
    r, o = L.add_sub(circ, 6, 3, 1)
    checks.append(("worked example: 6 - 3 = 0110 + 1100 + 1 = 10011, keep 4 bits 0011 = 3, no overflow",
                   0b0110 + 0b1100 + 1 == 0b10011 and int(r) == 3 and int(o) == 0 and (0b10011 & 15) == 3))

    # exhaustive add/subtract with overflow vs Python
    S = np.arange(-8, 8)
    SA, SB = [x.ravel() for x in np.meshgrid(S, S)]
    ok = True
    for sub in (0, 1):
        res, ovf = L.add_sub(circ, SA, SB, sub)
        true = SA - SB if sub else SA + SB
        wrapped = ((true + 8) % 16) - 8
        ok &= np.array_equal(res, wrapped) and np.array_equal(ovf, (true != wrapped).astype(int))
        ok &= all(L.python_reference(int(x), int(y), sub, 4) == (int(rr), int(oo)) for x, y, rr, oo in zip(SA, SB, res, ovf))
    checks.append(("4-bit adder-subtractor: result and overflow flag match Python for all 256 signed pairs, add and subtract", bool(ok)))
    r1, o1 = L.add_sub(circ, 7, 1, 0)
    r2, o2 = L.add_sub(circ, -8, 1, 1)
    checks.append(("failure case: 7 + 1 wraps to -8 with overflow flag 1; -8 - 1 wraps to 7 with flag 1",
                   int(r1) == -8 and int(o1) == 1 and int(r2) == 7 and int(o2) == 1))
    r3, o3 = L.add_sub(circ, -8, -8, 0)
    r4, o4 = L.add_sub(circ, 5, -3, 0)
    checks.append(("edge: -8 + -8 overflows to 0; 5 + -3 = 2 does not overflow",
                   int(r3) == 0 and int(o3) == 1 and int(r4) == 2 and int(o4) == 0))
    checks.append(("64 of the 256 signed pairs overflow when adding in 4 bits",
                   int(L.add_sub(circ, SA, SB, 0)[1].sum()) == 64))

    # gate delays
    d = {n: L.carry_delay(L.build_adder(n)) for n in (4, 8, 16)}
    checks.append(("carry-out delay is 9 gate delays for 4 bits, 17 for 8 bits, 33 for 16 bits", d == {4: 9, 8: 17, 16: 33}))
    checks.append(("carry-out delay = 2n + 1 for every width 1 to 32, and the last sum bit takes 2n for n >= 2",
                   all(L.carry_delay(L.build_adder(n)) == 2 * n + 1 for n in range(1, 33))
                   and all(L.sum_delay(L.build_adder(n)) == 2 * n for n in range(2, 33))))
    checks.append(("each extra bit adds exactly 2 gate delays", all(
        L.carry_delay(L.build_adder(n + 1)) - L.carry_delay(L.build_adder(n)) == 2 for n in range(1, 20))))
    checks.append(("data-dependent chain: 127 + 1 passes the carry through 7 stages, 255 + 1 through 8",
                   L.longest_carry_run(127, 1, 8) == 7 and L.longest_carry_run(255, 1, 8) == 8
                   and L.longest_carry_run(0, 0, 8) == 0))

    # ALU, every input pair and every op
    alu = L.build_alu(4)
    UA, UB = np.meshgrid(np.arange(16), np.arange(16))
    UA, UB = UA.ravel(), UB.ravel()
    ok = True
    for op in range(4):
        res, ovf = L.run_alu(alu, UA, UB, op)
        for x, y, rr, oo in zip(UA, UB, res, ovf):
            x, y = int(x), int(y)
            sx, sy = x - 16 if x >= 8 else x, y - 16 if y >= 8 else y
            if op == 0:
                t = sx + sy
                exp = (t % 16, int(not -8 <= t <= 7))
            elif op == 1:
                t = sx - sy
                exp = (t % 16, int(not -8 <= t <= 7))
            elif op == 2:
                exp = (x & y, 0)
            else:
                exp = (x | y, 0)
            ok &= (int(rr), int(oo)) == exp
    checks.append(("4-bit ALU (add, subtract, AND, OR) matches Python for all 256 pairs x 4 operations", bool(ok)))
    checks.append(("the ALU slice uses 59 gates", alu["c"].size() == 59))
    r, o = L.run_alu(alu, 7, 1, 0)
    checks.append(("ALU adds 7 + 1 to pattern 1000 (= -8) and raises overflow", int(r) == 8 and int(o) == 1))

    # seven-segment
    checks.append(("seven-segment: digit 8 lights all 7 segments, digit 1 lights 2, digits 0 to 9 light 49 in total",
                   sum(L.seven_segment(8)) == 7 and sum(L.seven_segment(1)) == 2
                   and sum(sum(L.seven_segment(k)) for k in range(10)) == 49))

    # mini-project: 8-bit adder-subtractor, every pair
    c8 = L.build_add_sub(8)
    S8 = np.arange(-128, 128)
    A8, B8 = [x.ravel() for x in np.meshgrid(S8, S8)]
    ok, counts = True, []
    for sub in (0, 1):
        res, ovf = L.add_sub(c8, A8, B8, sub)
        true = A8 - B8 if sub else A8 + B8
        wrapped = ((true + 128) % 256) - 128
        ok &= np.array_equal(res, wrapped) and np.array_equal(ovf, (true != wrapped).astype(int))
        counts.append(int(ovf.sum()))
    checks.append(("mini-project: 8-bit adder-subtractor matches Python for all 65536 pairs, add and subtract", bool(ok)))
    checks.append(("mini-project: 16384 pairs overflow for add and 16384 for subtract", counts == [16384, 16384]))
    checks.append(("mini-project: the 8-bit adder-subtractor has 49 gates and its overflow flag is ready after 19 gate delays",
                   c8["c"].size() == 49 and c8["c"].depth[c8["overflow"]] == 19))
    widest = max(n for n in range(2, 40)
                 if L.build_add_sub(n)["c"].depth[L.build_add_sub(n)["overflow"]] * 0.5 <= 10.0)
    checks.append(("mini-project: with 0.5 ns per gate and a 10 ns budget the widest adder-subtractor is 8 bits (19 x 0.5 = 9.5 ns; 9 bits needs 10.5 ns)",
                   widest == 8 and 19 * 0.5 == 9.5 and L.build_add_sub(9)["c"].depth[L.build_add_sub(9)["overflow"]] == 21))
    # numbers quoted in the text
    checks.append(("4 bits cover -8 to 7 and 8 bits cover -128 to 127",
                   int(L.from_twos(np.arange(16), 4).min()) == -8 and int(L.from_twos(np.arange(16), 4).max()) == 7
                   and int(L.from_twos(np.arange(256), 8).min()) == -128 and int(L.from_twos(np.arange(256), 8).max()) == 127))
    ad4 = L.build_add_sub(4)
    w = ad4["c"].run({**{f"a{i}": np.array([(7 >> i) & 1]) for i in range(4)}, **{f"b{i}": np.array([(1 >> i) & 1]) for i in range(4)}, "sub": np.array([0])})
    checks.append(("7 + 1: carry into the top bit is 1 and carry out of it is 0, so they differ",
                   int(w[ad4["carry"][2]][0]) == 1 and int(w[ad4["carry"][3]][0]) == 0))
    w2 = ad4["c"].run({**{f"a{i}": np.array([(15 >> i) & 1]) for i in range(4)}, **{f"b{i}": np.array([(1 >> i) & 1]) for i in range(4)}, "sub": np.array([0])})
    checks.append(("-1 + 1 (1111 + 0001) has a carry out of the top bit but no overflow",
                   int(w2[ad4["carry"][3]][0]) == 1 and int(w2[ad4["overflow"]][0]) == 0))
    r5, o5 = L.add_sub(ad4, 5, 7, 1)
    checks.append(("5 - 7: 0101 + 1000 + 1 = 1110 = 14, read as -2, no overflow",
                   0b0101 + 0b1000 + 1 == 14 == 0b1110 and int(r5) == -2 and int(o5) == 0))
    checks.append(("carry delay at 12 bits is 25; widest ripple adder with carry out within 20 ns at 0.5 ns per gate is 19 bits",
                   L.carry_delay(L.build_adder(12)) == 25
                   and max(n for n in range(1, 60) if L.carry_delay(L.build_adder(n)) * 0.5 <= 20) == 19
                   and L.carry_delay(L.build_adder(19)) == 39 and L.carry_delay(L.build_adder(20)) == 41))
    r6, o6 = L.add_sub(c8, 100, 100, 0)
    r7, o7 = L.add_sub(c8, 127, 1, 0)
    r8, o8 = L.add_sub(c8, 45, 19, 0)
    r9, o9 = L.add_sub(c8, -1, 1, 0)
    checks.append(("8 bits: 100 + 100 wraps to -56 (overflow); 127 + 1 = -128 (overflow); 45 + 19 = 64 and -1 + 1 = 0 (no overflow)",
                   (int(r6), int(o6)) == (-56, 1) and (int(r7), int(o7)) == (-128, 1)
                   and (int(r8), int(o8)) == (64, 0) and (int(r9), int(o9)) == (0, 0)))
    checks.append(("carry chain lengths: 45 + 19 passes a carry through 6 stages, 127 + 1 through 7, -1 + 1 (255 + 1) through 8",
                   L.longest_carry_run(45, 19, 8) == 6 and L.longest_carry_run(127, 1, 8) == 7 and L.longest_carry_run(255, 1, 8) == 8))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
